# 9. Protein Sequence Analysis

The programs of this chapter, in the order the book prints
them. Each cell holds the file of the same name in this folder,
so editing a cell does not change the script beside it.

Run the cells from this folder, where the data files the chapter
uses live. Some cells call command line tools or download data,
and a few take minutes.


## `sickle_charge.py`

Normal and sickle beta globin compared by what their sequences predict.


In [ ]:
# Normal and sickle beta globin compared by what their sequences predict.
from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis

beta = str(SeqIO.read("hbb_human.fasta", "fasta").seq)
mature = beta[1:]  # the cell removes the first methionine
sickle = mature[:5] + "V" + mature[6:]  # glutamic acid 6 becomes valine

for name, protein in (("normal", mature), ("sickle", sickle)):
    analysis = ProteinAnalysis(protein)
    print(name)
    print(f"  length              {len(protein):8}")
    print(f"  molecular weight    {analysis.molecular_weight():8.1f} Da")
    print(f"  isoelectric point   {analysis.isoelectric_point():8.2f}")
    print(f"  charge at pH 7.4    {analysis.charge_at_pH(7.4):8.2f}")
    print(f"  GRAVY               {analysis.gravy():8.3f}")

## `hydropathy.py`

Find the membrane crossing helices of human rhodopsin (UniProt P08100) from its sequence alone, with a Kyte-Doolittle hydropathy plot.


In [ ]:
# Find the membrane crossing helices of human rhodopsin (UniProt P08100)
# from its sequence alone, with a Kyte-Doolittle hydropathy plot.
import pandas as pd
from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio.SeqUtils.ProtParamData import kd

WINDOW = (
    19  # about the number of residues a helix needs to cross a membrane
)
THRESHOLD = 1.6  # Kyte and Doolittle's cut-off for a membrane segment
MAX_GAP = 10  # dips shorter than this do not split a segment

rhodopsin = str(SeqIO.read("rhodopsin.fasta", "fasta").seq)
scores = ProteinAnalysis(rhodopsin).protein_scale(kd, WINDOW)
centers = range(WINDOW // 2 + 1, len(rhodopsin) - WINDOW // 2 + 1)
above = [p for p, score in zip(centers, scores) if score > THRESHOLD]

# Join positions above the threshold into segments, bridging short dips
segments = [[above[0], above[0]]]
for position in above[1:]:
    if position - segments[-1][1] <= MAX_GAP:
        segments[-1][1] = position
    else:
        segments.append([position, position])

known = pd.read_csv("rhodopsin_helices.tsv", sep="\t", comment="#")
print(f"{len(rhodopsin)} residues, {len(segments)} predicted segments")
for first, last in segments:
    overlap = known[(known["start"] <= last) & (known["end"] >= first)]
    helix = overlap["helix"].iloc[0] if len(overlap) else "none"
    print(f"  centers {first:3} to {last:3}   UniProt helix {helix}")

for helix in known.itertuples():
    inside = [
        s
        for p, s in zip(centers, scores)
        if helix.start <= p <= helix.end
    ]
    print(
        f"helix {helix.helix}: {helix.start:3} to {helix.end:3}, "
        f"highest score {max(inside):.2f}"
    )

## `domains.py`

What InterPro knows about the families and domains of two proteins.


In [ ]:
# What InterPro knows about the families and domains of two proteins.
import json
import urllib.request

INTERPRO = (
    "https://www.ebi.ac.uk/interpro/api/entry/interpro/protein/uniprot"
)

for accession, name in (
    ("P68871", "beta globin"),
    ("P08100", "rhodopsin"),
):
    with urllib.request.urlopen(f"{INTERPRO}/{accession}/") as response:
        entries = json.load(response)["results"]
    print(f"{name} ({accession}): {len(entries)} InterPro entries")
    for entry in entries:
        meta = entry["metadata"]
        pieces = [
            piece
            for location in entry["proteins"][0][
                "entry_protein_locations"
            ]
            for piece in location["fragments"]
        ]
        start = min(piece["start"] for piece in pieces)
        end = max(piece["end"] for piece in pieces)
        name = meta["name"][:26]
        print(f"  {meta['accession']}  {start:3} to {end:3}  {name}")